# CRITCLOUD — corrected experiments (run top to bottom)

**Before you start:** put this notebook in the same folder as `critcloud_core.py`, `simulator.py`, the other `.py` files from the zip, and `qws.csv`.

Run each cell with **Shift + Enter**, in order.

## Step 1 — Install / check libraries
Run once. If everything is already installed, it finishes quickly.

In [ ]:
%pip install -q numpy pandas scipy matplotlib

## Step 2 — Check the files are in the folder
Every line should say `OK`. If one says `MISSING`, copy that file into this folder.

In [ ]:
import os
for f in ["qws.csv","critcloud_core.py","simulator.py","test_reproduce.py",
          "exp_paper_tables.py","run_simulation.py","analysis_extra.py",
          "exp_bwm_perturbation.py","bwm_revision_options.py","exp_alpha.py",
          "exp_normalization.py","check_rie_theory.py"]:
    print(("OK      " if os.path.exists(f) else "MISSING ") + f)

## Step 3 — Safety test
Checks that the new code reproduces your old notebook numbers. Expected output: `ALL REPRODUCTION CHECKS PASSED`.

In [ ]:
%run test_reproduce.py

## Step 4 — Re-run all real-data (QWS) tables
Takes about 30 seconds. Saves tables T1–T8 in the `results/` folder.

In [ ]:
%run exp_paper_tables.py

## Step 5 — Look at the key real-data tables

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
print("T1  BWM weights and consistency (CR_I_j > ~0.2 = inconsistent judgement)")
display(pd.read_csv("results/T1_bwm_consistency.csv", index_col=0))
print("T2  Weights of all methods")
display(pd.read_csv("results/T2_weights.csv", index_col=0))
print("T3  Redundancy diagnostics")
display(pd.read_csv("results/T3_redundancy.csv", index_col=0))

In [ ]:
print("T4  Kendall tau vs Equal-TOPSIS and vs SLA reference (full data)")
display(pd.read_csv("results/T4_agreement_full.csv", index_col=0))
print("T6  Circularity: the closer to equal weights, the higher tau vs Equal-TOPSIS")
display(pd.read_csv("results/T6_circularity.csv", index_col=0))
print("T7  Pairwise rank-reversal rate (%)")
display(pd.read_csv("results/T7_rank_reversal.csv", index_col=0))
print("T8  Skewness and max-min compression")
display(pd.read_csv("results/T8_skew.csv", index_col=0))

## Step 5b — BWM perturbation test on real data (Reviewer 1)
Changes the BWM answers by ±1 and ±2, 1000 times each, and measures how much each method's ranking moves. About 35 seconds. 1.0 = ranking unchanged.

In [ ]:
%run exp_bwm_perturbation.py

## Step 5c — BWM consistency check and corrected vector options
Shows the consistency ratio against the official threshold, the ordinal conflicts, and minimal-change corrected vectors. **The supervisor chooses one option.**

In [ ]:
%run bwm_revision_options.py

## Step 6 — Run the simulation study (known true ranking)
**This is the slow step: about 5–6 minutes.** Wait until the star `[*]` next to the cell turns into a number.

Tip: for a quick trial, first open `run_simulation.py` and change `REPS = 500` to `REPS = 50` (about 30 seconds), then set it back to 500 for the final paper results.

In [ ]:
%run run_simulation.py

## Step 7 — Follow-up analysis
Answers: does CRITIC help beyond simple shrinkage? Does skewed data hurt? Does low RIE mean better ranking?

In [ ]:
%run analysis_extra.py

## Step 7b — Choosing alpha (Week 2)
Real-data trade-off curve and the best alpha by judgement noise and redundancy. About 1 minute.

In [ ]:
%run exp_alpha.py

## Step 7c — Skewed data: which normalisation? (Week 2)
Compares max-min, log + max-min and rank normalisation. About 45 seconds.

In [ ]:
%run exp_normalization.py

## Step 7d — RIE theory check (Week 2)
Verifies the propositions in `RIE_theory.tex` and prints the RIE = concentration x redundancy table.

In [ ]:
%run check_rie_theory.py

## Step 8 — Key simulation tables

In [ ]:
print("S2  Mean Kendall tau to the TRUE ranking")
display(pd.read_csv("results/S2_tau_by_structure.csv", index_col=[0,1]).T)
print("S5  Friedman mean rank (1 = best)")
display(pd.read_csv("results/S5_friedman_mean_ranks.csv", index_col=0).sort_values("dirichlet"))
print("S4  CRITCLOUD(0.5) vs every other method (users with real preferences)")
display(pd.read_csv("results/S4_wilcoxon_dirichlet.csv"))
print("S7  Does CRITIC add value beyond blending with equal weights?")
display(pd.read_csv("results/S7_critic_vs_shrinkage.csv"))
print("S6  Effect of noise in user BWM judgements")
display(pd.read_csv("results/S6_pref_noise.csv", index_col=0).T)

## Step 9 — Figures for the paper
Saves PNG files in `figures/`.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
os.makedirs("figures", exist_ok=True)

# Figure 1: circularity of the equal-weight reference
d = pd.read_csv("results/T6_circularity.csv", index_col=0)
fig, ax = plt.subplots(figsize=(5.5,4))
ax.scatter(d["L2_to_equal"], d["tau_vs_Equal"])
for m,r in d.iterrows(): ax.annotate(m, (r["L2_to_equal"], r["tau_vs_Equal"]), fontsize=8, xytext=(4,3), textcoords="offset points")
ax.set_xlabel("Distance of weight vector from equal weights (L2)"); ax.set_ylabel("Kendall tau vs Equal-TOPSIS")
ax.set_title("Equal-weight reference rewards closeness to uniform weights", fontsize=10)
fig.tight_layout(); fig.savefig("figures/fig_circularity.png", dpi=300); plt.show()

# Figure 2: accuracy vs BWM judgement noise
n = pd.read_csv("results/S6_pref_noise.csv", index_col=0)
fig, ax = plt.subplots(figsize=(6,4))
for m in ["BWM","CRITCLOUD(0.5)","BWM+Equal(0.5)","CRITIC","Equal","Oracle (best linear)"]:
    ax.plot(n.index, n[m], marker="o", label=m)
ax.set_xlabel("Noise in user judgements (sigma)"); ax.set_ylabel("Kendall tau to true ranking")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig("figures/fig_pref_noise.png", dpi=300); plt.show()

# Figure 3: mean tau by redundancy level
s = pd.read_csv("results/S2_tau_by_structure.csv", index_col=[0,1])
sel = ["BWM","CRITCLOUD(0.5)","BWM+Equal(0.5)","CRITIC","Equal","OPTCLOUD"]
fig, axes = plt.subplots(1,2, figsize=(10,4), sharey=True)
for ax, th in zip(axes, ["dirichlet","uniform"]):
    sub = s.loc[th].reindex(["none","moderate","high"])[sel]
    sub.plot(kind="bar", ax=ax, rot=0, legend=False)
    ax.set_title("Users with strong preferences" if th=="dirichlet" else "All factors equally important")
    ax.set_xlabel("Redundancy level"); ax.set_ylabel("Kendall tau to true ranking")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig("figures/fig_tau_by_redundancy.png", dpi=300); plt.show()

# Figure 4: alpha trade-off on QWS
a = pd.read_csv("results/A1_alpha_tradeoff_qws.csv")
fig, ax1 = plt.subplots(figsize=(6.5,4))
ax1.plot(a["alpha"], a["tau_to_user_BWM"], marker="o", label="Agreement with user's BWM ranking (tau)")
ax1.plot(a["alpha"], a["top1_stable_%"]/100, marker="s", label="Same best provider under BWM errors (share)")
ax1.set_xlabel("alpha (0 = user only, 1 = CRITIC only)"); ax1.set_ylabel("Value")
ax1.legend(fontsize=8, loc="lower left"); ax1.set_title("Choosing alpha on QWS: fidelity vs stability", fontsize=10)
fig.tight_layout(); fig.savefig("figures/fig_alpha_tradeoff.png", dpi=300); plt.show()
print("Saved figures in the figures/ folder")

## Done — what to check
- Step 3 printed **ALL REPRODUCTION CHECKS PASSED**
- `results/` has T1–T8 and S0–S9
- `figures/` has four PNG files

These tables replace the old Tables 6–7 and the old τ / Friedman figures in the paper.